#### ResNet50 모델을 이용한 전의학습

In [3]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

- 데이터 경로 설정

In [4]:
base_dir = './cats_and_dogs_filtered/'
train_dir= os.path.join(base_dir, 'train')
valid_dir= os.path.join(base_dir, 'validation')

- 데이터 제너레이터 설정(ResNet50 전용 전처리 포함)

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,   # ResNet/VGG 계열에 맞는 입력 전처리(정규화 및 표준화)
    rotation_range=40,      # 램던 회전
    width_shift_range=0.2,  # 가로 이동
    height_shift_range=0.2, # 세로 이동
    shear_range=0.2,        # 기울기 변경
    zoom_range=0.2,         # 축소/확대
    horizontal_flip=True,   # 좌우 반전
    fill_mode='nearest'     # 빈공간 채우기
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # ResNet 계열의 권장 사이즈 => 224*224
    batch_size=32,
    class_mode='binary')

valid_generator = train_datagen.flow_from_directory(
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


- 사전 학습된 ResNet50 로드

In [7]:
resnet_base = ResNet50(weights='imagenet',
                       input_shape=(224, 224, 3),
                       include_top=False)   # 분류기는 제거

I0000 00:00:1773819948.773006  125749 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5528 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


- 특징 추출부 동결

In [ ]:
resnet_base.trainable = False   # ResNet의 기본 모델의 모든 층을 학습하지 않도록 고정(가중치 업데이트 금지)


- 모델 구성

In [9]:
model = models.Sequential([
    resnet_base,
    layers.GlobalAveragePooling2D(),  # Flatten 레이어 대신 사용(연산 효율이 좋은 전역 평균 폴링 사용)
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

- 모델 컴파일

In [12]:
model.compile(
    optimizer=optimizers.Adam(0.0001),    
    loss = 'binary_crossentropy',
    metrics=['acc']
)

- 학습하기

In [ ]:
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=valid_generator,
    verbose=1
)

Epoch 1/30


I0000 00:00:1773820667.109687  125749 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1773820685.921229  129193 service.cc:153] XLA service 0x777b84063f00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773820685.921276  129193 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1773820686.436308  129193 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773820689.724800  129193 cuda_dnn.cc:461] Loaded cuDNN version 91002
I0000 00:00:1773820690.364909  129193 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_43749__.390
I0000 00:00:1773820712.741515  129193 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the 

 7/63 ━━━━━━━━━━━━━━━━━━━━ 12s 229ms/step - acc: 0.6365 - loss: 0.6394

I0000 00:00:1773820718.624753  129190 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_43749__.390


63/63 ━━━━━━━━━━━━━━━━━━━━ 97s 823ms/step - acc: 0.9370 - loss: 0.1631 - val_acc: 0.9460 - val_loss: 0.2379
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 285ms/step - acc: 0.9725 - loss: 0.0817 - val_acc: 0.9700 - val_loss: 0.1102
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 316ms/step - acc: 0.9780 - loss: 0.0626 - val_acc: 0.9740 - val_loss: 0.0785
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 302ms/step - acc: 0.9875 - loss: 0.0345 - val_acc: 0.9810 - val_loss: 0.0536
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 284ms/step - acc: 0.9855 - loss: 0.0388 - val_acc: 0.9180 - val_loss: 0.3298
Epoch 6/30
26/63 ━━━━━━━━━━━━━━━━━━━━ 7s 207ms/step - acc: 0.9847 - loss: 0.0409

In [ ]:
import matplotlib.pyplot as plt
his_data = history.history

loss = his_data['loss']
val_loss = his_data['val_loss']

epochs = range(1, len(loss)+1)

fig = plt.figure(figsize=(10, 5))

# 훈련 및 검증 손실율(lossl)
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(epochs, loss, color='blue', label='Train_loss')
ax1.plot(epochs, val_loss, color='orange', label='Val_loss')
ax1.set_title('Train & Validation Loss')
ax1.set_xlabel('epochs')
ax1.set_ylabel('loss')
ax1.legend()

# 훈련 및 검증 정확도
acc = his_data['acc']
val_acc = his_data['val_acc']

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(epochs, acc, color='blue', label='Train_acc')
ax2.plot(epochs, val_acc, color='orange', label='Val_acc')
ax2.set_title('Train & Validation Acc')
ax2.set_xlabel('epochs')
ax2.set_ylabel('acc')
ax2.legend()

plt.show()